### 🎯 [미션] 주석이 달린 줄의 None 또는 _______ 부분을 채워 코드를 완성하세요.

각 셀을 순서대로 실행(Shift + Enter)해야 합니다.

---

#### 1️⃣ 필요한 라이브러리 설치 및 불러오기

In [ ]:
# Hugging Face transformers 및 관련 라이브러리 설치
!pip install transformers pillow torch torchvision accelerate -q

In [ ]:
import os
from tabulate import tabulate

import torch
import glob
from PIL import Image
from torch.utils.data import Dataset as TorchDataset
from transformers import (
    AutoImageProcessor,
    AutoModelForImageClassification,
    TrainingArguments,
    Trainer
)
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from pathlib import Path

print("✅ 라이브러리 로드 완료")
print(f"PyTorch 버전: {torch.__version__}")
print(f"CUDA 사용 가능: {torch.cuda.is_available()}")

---
#### 2️⃣ 경로 설정

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd "/content/drive/MyDrive/2026_AI_Advanced_Study-main/4차시/06_car_damage_classification/code"

In [ ]:
# Google Drive 내 작업 디렉토리 경로 설정
WORK_DIR = '/content/drive/MyDrive/2026_AI_Advanced_Study-main/4차시/06_car_damage_classification'
WORK_DIR = '/home/nute11a/workspace/2026_AI_Advanced_Study/4차시/06_car_damage_classification'

# 작업 디렉토리로 이동
os.chdir(WORK_DIR)
print(f"현재 작업 디렉토리: {WORK_DIR}")

# 데이터 경로 확인
DATA_DIR = Path(f'{WORK_DIR}/data')
if DATA_DIR.exists():
    print(f"✅ 데이터 디렉토리 발견: {DATA_DIR}")
else:
    print(f"❌ 데이터 디렉토리를 찾을 수 없습니다: {DATA_DIR}")

---

#### 3️⃣ 데이터셋 경로 및 규모 확인

모델을 학습시키기 전, **준비된 데이터가 총 몇 장인지** 확인하는 과정은 필수입니다.

YOLO Classification은 Detection과 달리 **config.yaml 파일이 필요 없습니다.**  
폴더 구조 자체가 클래스 정보를 정의합니다:
```
data/
├── train/
│   ├── 01_dent/
│   ├── 02_scratch/
│   └── ...
├── val/
└── test/
```

In [ ]:
# 🎯 [미션] 데이터 폴더의 경로를 설정하세요.
# 힌트: 상위 폴더의 data 디렉토리를 가리킵니다.
DATA_PATH = 'data'

print(f"📘 데이터 경로: {DATA_PATH}")
print("=" * 70)

# 클래스별 데이터 수집
class_data = {}

for split in ['train', 'val', 'test']:
    split_path = os.path.join(DATA_PATH, split)
    if os.path.exists(split_path):
        classes = [d for d in os.listdir(split_path) if os.path.isdir(os.path.join(split_path, d))]
        
        for cls in classes:
            if cls not in class_data:
                class_data[cls] = {'train': 0, 'val': 0, 'test': 0}
            
            cls_path = os.path.join(split_path, cls)
            image_count = len(glob.glob(os.path.join(cls_path, '*.[jJ][pP][gG]'))) + \
                          len(glob.glob(os.path.join(cls_path, '*.[pP][nN][gG]')))
            class_data[cls][split] = image_count

# 표 데이터 생성
table_data = []
train_total = val_total = test_total = 0

for cls in sorted(class_data.keys()):
    train = class_data[cls]['train']
    val = class_data[cls]['val']
    test = class_data[cls]['test']
    total = train + val + test
    
    table_data.append([cls, train, val, test, total])
    train_total += train
    val_total += val
    test_total += test

# 합계 행 추가
table_data.append(['━' * 10, '━' * 5, '━' * 5, '━' * 5, '━' * 5])
table_data.append(['TOTAL', train_total, val_total, test_total, train_total + val_total + test_total])

# 표 출력
headers = ["클래스", "Train", "Val", "Test", "Total"]
print(tabulate(table_data, headers=headers, tablefmt="grid"))

# 비율 정보
print("\n📊 데이터셋 비율:")
total = train_total + val_total + test_total
print(f"   Train: {train_total:,}장 ({train_total/total*100:.1f}%)")
print(f"   Val:   {val_total:,}장 ({val_total/total*100:.1f}%)")
print(f"   Test:  {test_total:,}장 ({test_total/total*100:.1f}%)")
print(f"   Total: {total:,}장")


---
#### 4️⃣ 커스텀 데이터셋 클래스 정의


In [ ]:
class CarDmgDataset(TorchDataset):
    """로컬 이미지 파일을 로드하는 커스텀 데이터셋"""
    
    def __init__(self, data_dir, processor=None):
        self.data_dir = Path(data_dir)
        self.processor = processor
        self.samples = []
        
        # 클래스별 이미지 수집
        for cls_info in os.listdir(self.data_dir):
            cls_idx, cls_name = cls_info.split('_',1)
            cls_idx = int(cls_idx)
            for img_path in sorted((self.data_dir / cls_info).glob('*.jpg')):
                self.samples.append((img_path, cls_idx))
            
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        
        # 🎯 [미션] 이미지 파일을 열어서(open) RGB로 변환하세요
        image = Image._____(img_path).convert('RGB')
        
        # 전처리 적용
        if self.processor is not None:
            
            inputs = self.processor(images=image, return_tensors='pt')
            # 배치 차원 제거
            pixel_values = inputs['pixel_values'].squeeze(0)
            return {'pixel_values': pixel_values, 'label': label}
        else:
            return {'image': image, 'label': label}

print("✅ 커스텀 데이터셋 클래스 정의 완료")

---
#### 5️⃣ 이미지 프로세서 로드

위에서 설정한 모델 크기에 맞춰 YOLO 모델을 초기화합니다.

In [ ]:
# 사전학습된 모델의 설정과 동일한 Image Processor를 불러옵니다.
MODEL_NAME = "google/mobilenet_v2_1.0_224"

print(f"📦 이미지 프로세서 로드: {MODEL_NAME}")
# 🎯 [미션] AutoImageProcessor에서 사전학습된 이미지 프로세서를 불러오는 메서드를 완성하세요
processor = AutoImageProcessor._______(MODEL_NAME)

print("✅ 이미지 프로세서 로드 완료!")

---
#### 6️⃣ 데이터셋 로드

In [ ]:
print("📥 로컬 데이터셋 로드 중...\n")

# 데이터셋 생성
# 🎯 [미션] 학습/검증 데이터셋을 정의하세요
train_dataset = _______('data/train', processor=processor)
val_dataset = _______('data/val', processor=processor)

print(f"✅ 데이터셋 로드 완료!")
print(f"  - Train: {len(train_dataset)}장")
print(f"  - Val: {len(val_dataset)}장")

---
#### 7️⃣ 모델 로드 및 설정
Hugging Face의 사전학습된 MobileNetV2 모델을 로드하고, 클래스 분류를 위해 마지막 레이어를 조정합니다.

In [ ]:
# 사전학습된 모델 로드
print(f"🤖 모델 로드: {MODEL_NAME}")

model = AutoModelForImageClassification.from_pretrained(
    MODEL_NAME,
    num_labels=_______,  # 🎯 [미션] 분류할 클래스(파손 유형)의 개수를 입력하세요
    id2label={i: f"Class {i}" for i in range(6)},
    label2id={f"Class {i}": i for i in range(6)},
    ignore_mismatched_sizes=True
)

print(f"✅ 모델 로드 완료!")
print(f"모델 파라미터 수: {sum(p.numel() for p in model.parameters())/1e6:.2f}M")

---
#### 8️⃣ 학습 설정

In [ ]:
# 평가 메트릭 정의
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    
    accuracy = accuracy_score(labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average='weighted', zero_division=0
    )
    
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }

# 학습 설정
training_args = TrainingArguments(
    output_dir='runs/classification',
    num_train_epochs = _______,  # 🎯 [미션] 전체 데이터를 10-20번 반복해서 학습하도록 숫자를 채우세요
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=1e-5,
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    logging_dir='runs/classification/logs',
    logging_strategy='epoch',  # Epoch 단위로 로그 기록
    remove_unused_columns=False,
    push_to_hub=False,
    report_to='none'
)

print("=== 학습 설정 ===")
print(f"Epochs: {training_args.num_train_epochs}")
print(f"Batch Size: {training_args.per_device_train_batch_size}")
print(f"결과 저장 경로: {training_args.output_dir}")

---
#### 9️⃣ 모델 학습

In [ ]:
# Trainer 초기화
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=_______,  # 🎯 [미션] 위에서 준비한 학습 데이터셋 변수명을 넣어주세요
    eval_dataset=_______,   # 🎯 [미션] 위에서 준비한 검증 데이터셋 변수명을 넣어주세요
    compute_metrics=compute_metrics,
)

print("🚀 학습 시작...\n")
# 🎯 [미션] 훈련(train)을 시작하는 명령어를 입력하세요
train_result = trainer._______()

# 학습된 모델을 지정된 경로에 저장(save_model)합니다.
trainer.save_model('runs/classification/final_model')